## Поиск коллокаций в тексте

Коллокации - это неслучайные сочетания слов, всречающиеся в тексте. Они чаще других являются кандидатами на неоднословные термины предметной области. 

Для начала попробуем извлечь их при помощи Tf*Idf.

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import pymorphy3
import re
import numpy as np
from tqdm import tqdm

In [2]:
morph = pymorphy3.MorphAnalyzer()

In [3]:
with open("data/lenta2018.txt", encoding="utf-8") as news_file: # Файл с новостями.
    lenta_news = [n.split("-----\n")[1] for n in news_file.read().split("=====\n")[1:]]
    
with open("data/habr_10000.txt", encoding="utf-8") as news_file: # Файл с новостями.
    habr_news = [n for n in news_file.read().split("=====\n")[1:]]
    

In [4]:
imp_POS = ['ADJF', 'ADJS', 'NOUN', 'VERB', 'PRTF', 'PRTS', 'GRND', 'PREP']


def normalizePymorphy(text):
    tokens = re.findall('[а-яёА-ЯЁ]+-[а-яёА-ЯЁ]+-[а-яёА-ЯЁ]+|[а-яёА-ЯЁ]+-[а-яёА-ЯЁ]+|[а-яёА-ЯЁ]+', text)
    words = []
    for t in tokens:
        pv = morph.parse(t)
        if str(pv[0].tag.POS) in imp_POS:
            words.append(pv[0].normal_form)
    return words    


In [5]:
lenta_news = [' '.join(normalizePymorphy(news)) for news in tqdm(lenta_news)]
habr_news = [' '.join(normalizePymorphy(news)) for news in tqdm(habr_news)]

100%|█████████████████████████████████████████| 708/708 [00:13<00:00, 52.58it/s]


In [6]:
tfv = TfidfVectorizer(token_pattern="[а-яёА-ЯЁ]+-[а-яёА-ЯЁ]+-[а-яёА-ЯЁ]+|[а-яёА-ЯЁ]+-[а-яёА-ЯЁ]+|[а-яёА-ЯЁ]+", 
                      ngram_range=(2, 3))

In [7]:
tf_lenta = tfv.fit_transform(lenta_news[:100])

In [8]:
tf_lenta

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 26711 stored elements and shape (100, 24175)>

In [9]:
list(tfv.vocabulary_.items())[:10]

[('испытанный украина', 7636),
 ('украина первый', 22208),
 ('первый собственный', 13753),
 ('собственный крылатый', 19638),
 ('крылатый ракета', 8988),
 ('ракета создать', 17193),
 ('создать кб', 19793),
 ('кб луч', 8055),
 ('луч в', 9384),
 ('в рамка', 2208)]

In [10]:
# for news in tf_lenta:
#     print(sorted(news[0][:].todense()[0][0])[-5:])
    
    
freqwords = []
for i in tqdm(range(100)):

    tfs = [(word, tf_lenta[i, index]) for word, index in tfv.vocabulary_.items()
         if tf_lenta[i, index] != 0]
    fw = [w for w, f in sorted(tfs, key = lambda x: x[1], reverse = True)[:5]]
    freqwords.append(fw)


100%|█████████████████████████████████████████| 100/100 [00:26<00:00,  3.77it/s]


In [12]:
freqwords[:20]

[['ракета м',
  'собственный крылатый',
  'крылатый ракета',
  'противокорабельный ракета',
  'м комплекс'],
 ['на корточки',
  'русский гопник',
  'русский мат',
  'пользователь выразить',
  'выразить симпатия'],
 ['сильный удар',
  'в результат',
  'штат вирджиния',
  'вирджиния пассажирский',
  'пассажирский поезд'],
 ['в франция',
  'евро рубль',
  'франция открыться',
  'открыться первый',
  'первый бордель'],
 ['процент россиянин',
  'данный показатель',
  'процент избиратель',
  'на выборы',
  'президент собираться'],
 ['задержать звезда',
  'звезда телесериал',
  'телесериал детектив',
  'детектив алексей',
  'алексей насонов'],
 ['на ближний',
  'в регион',
  'совместный предприятие',
  'клиентский сервис',
  'доставка в'],
 ['причина исчезновение',
  'исчезновение рыболовецкий',
  'рыболовецкий судный',
  'судный восток',
  'восток с'],
 ['в сексуальный',
  'сексуальный домогательство',
  'в сексуальный домогательство',
  'с оральный',
  'оральный секс'],
 ['американский зооз

Из теории вероятности используется несколько формул для расчета степени неслучайности для словосочетий.

$$MI(x, y)=log(\frac{f(x,y)*N}{f(x)*f(y)}),$$ 
где $f(x)$ и $f(y)$ - частоты встречаемости слов $x$ и $y$, а $f(x,y)$ - частота встречаемости фразы $x y$.

Для данной метрики берутся слова с самым большими значениями. Однако, так как нас интересует скорее ранжирование слов, чем абсолютные значения метрики, на практике удобнее использовать редуцированную формулу.

$$PMI(x, y)=\frac{f(x,y)}{f(x)*f(y)}$$

Еще одна формула:

$$t-score(x, y)=\frac{f(x,y)-\frac{f(x)*f(y)}{N}}{\sqrt{f(x,y)}},$$
где N - количество слов в коллекции.

О различиях между результатами можно прочитать [здесь](https://cyberleninka.ru/article/n/ot-kollokatsiy-k-konstruktsiyam/viewer) (можно начинать со второго раздела) или [здесь](https://www.researchgate.net/publication/340371729_K_voprosu_o_shodstve_mer_associacii_primenitelno_k_zadace_avtomaticeskogo_izvlecenia_glagolnyh_kollokacij).

Еще можно посмотреть информацию о [c-value](https://www.researchgate.net/publication/220387502_Automatic_Recognition_of_Multi-word_Terms_The_C-value_NC-value_Method).

Давайте посчитаем значения этих 

In [13]:
cntv1 = CountVectorizer(token_pattern="[а-яёА-ЯЁ]+-[а-яёА-ЯЁ]+-[а-яёА-ЯЁ]+|[а-яёА-ЯЁ]+-[а-яёА-ЯЁ]+|[а-яёА-ЯЁ]+", 
                        ngram_range=(1, 1))
cntv2 = CountVectorizer(token_pattern="[а-яёА-ЯЁ]+-[а-яёА-ЯЁ]+-[а-яёА-ЯЁ]+|[а-яёА-ЯЁ]+-[а-яёА-ЯЁ]+|[а-яёА-ЯЁ]+", 
                        ngram_range=(2, 2))

In [14]:
cnt_lenta_1 = cntv1.fit_transform(['\n'.join(lenta_news)])
cnt_lenta_2 = cntv2.fit_transform(['\n'.join(lenta_news)])

In [15]:
def calc_MI_tscore(cnt_lenta_1, cnt_lenta_2, cntv1, cntv2, thr):
    mis = {}
    t_scores = {}
    corp_len = cnt_lenta_1.sum()
    for pair, index2 in tqdm(cntv2.vocabulary_.items()):
        freq12 = cnt_lenta_2[0, index2]
        if freq12 < thr:
            continue
        words = pair.split(' ')
        freq1 = cnt_lenta_1[0, cntv1.vocabulary_[words[0]]]
        freq2 = cnt_lenta_1[0, cntv1.vocabulary_[words[1]]]
        mis[pair] = freq12 / (freq1 * freq2)
        t_scores[pair] = (freq12 - (freq1 * freq2) / corp_len) / np.sqrt(freq12)
    return mis, t_scores

In [16]:
mis, t_scores = calc_MI_tscore(cnt_lenta_1, cnt_lenta_2, cntv1, cntv2, 10)

100%|█████████████████████████████████| 139120/139120 [00:50<00:00, 2769.13it/s]


In [17]:
sorted(mis.items(), key=lambda x: x[1], reverse=True)[:100]

[('январярусский смертьзакать', np.float64(0.1)),
 ('январясадизм какой-тобдсм-маска', np.float64(0.1)),
 ('февралязолоть стволыонить', np.float64(0.1)),
 ('упалокто ронять', np.float64(0.08333333333333333)),
 ('демьян кудрявцев', np.float64(0.08333333333333333)),
 ('возрождать сверхзвуковой', np.float64(0.07692307692307693)),
 ('рен тв', np.float64(0.07692307692307693)),
 ('сверхзвуковой ракетоносец', np.float64(0.06993006993006994)),
 ('рекс тиллерсон', np.float64(0.06666666666666667)),
 ('раюдин юсуфов', np.float64(0.06666666666666667)),
 ('шахабас шахов', np.float64(0.06666666666666667)),
 ('исаев раюдин', np.float64(0.0625)),
 ('какой-тобдсм-маска суровый', np.float64(0.058823529411764705)),
 ('февралякаждый своероссия', np.float64(0.058823529411764705)),
 ('оливковый ветвь', np.float64(0.05803571428571429)),
 ('суровый госпожа', np.float64(0.053475935828877004)),
 ('алина загитов', np.float64(0.04807692307692308)),
 ('шамиль исаев', np.float64(0.046875)),
 ('саудовский аравия', n

In [18]:
sorted(t_scores.items(), key=lambda x: x[1], reverse=True)[:100]

[('по тема', np.float64(29.034779342452346)),
 ('материал по', np.float64(28.971661083100336)),
 ('один из', np.float64(20.641832991948036)),
 ('по слово', np.float64(19.904789227329328)),
 ('в год', np.float64(18.689342637493702)),
 ('о сообщать', np.float64(18.03641982032943)),
 ('в время', np.float64(15.082028065512983)),
 ('в пхенчхан', np.float64(13.183645299788584)),
 ('по данные', np.float64(12.830766146341503)),
 ('олимпийский игра', np.float64(12.62361323219482)),
 ('на сайт', np.float64(12.494492351663348)),
 ('игра в', np.float64(11.603844664385685)),
 ('в тот', np.float64(11.463002451742444)),
 ('российский спортсмен', np.float64(11.329912708166699)),
 ('в частность', np.float64(11.327931558862113)),
 ('в россия', np.float64(11.074194123249194)),
 ('владимир путин', np.float64(10.893433935592524)),
 ('тот число', np.float64(10.878861844865753)),
 ('в результат', np.float64(10.863932539111818)),
 ('ссылка на', np.float64(10.85465083430653)),
 ('участие в', np.float64(10.8496

Несколько странные слова в топе в самом деле есть в тексте. Просто надо было при морфологическом анализе посмотреть, что это предсказанные слова, а не словарные.

Вообще, появление этих слов связано с тем, что MI поощряет появление редких слов. Например, если есть два слова, встретившихся вместе 1 раз, причем каждое слово встретилось только один раз (то есть они встретились только вместе), то MI = 1 / 1 * 1 = 1, то есть максимальному значению. Если есть два слова, встречающиеся только вместе, но 100 раз, MI = 100 / 100 * 100 = 0.01.

Посмотрим какие словосочетания встречаются больше 100 раз и какие у них получаются значения метрик.

In [19]:
mis_100, t_scores_100 = calc_MI_tscore(cnt_lenta_1, cnt_lenta_2, cntv1, cntv2, 100)

100%|█████████████████████████████████| 139120/139120 [00:50<00:00, 2732.05it/s]


In [20]:
sorted(mis_100.items(), key=lambda x: x[1], reverse=True)[:50]

[('риа новость', np.float64(0.004545071963639425)),
 ('владимир путин', np.float64(0.0030199213297804847)),
 ('уголовный дело', np.float64(0.0020567667626491155)),
 ('олимпийский комитет', np.float64(0.00100116499199068)),
 ('кроме тот', np.float64(0.0008683437619716511)),
 ('олимпийский игра', np.float64(0.0008260985571496008)),
 ('тот число', np.float64(0.000612948537862342)),
 ('российский спортсмен', np.float64(0.00041872834439287586)),
 ('тема декабрь', np.float64(0.00037104692316474483)),
 ('один из', np.float64(0.00036986138681053936)),
 ('с ссылка', np.float64(0.00026631036176885176)),
 ('по тема', np.float64(0.00026117928293542726)),
 ('материал по', np.float64(0.0002542138582869032)),
 ('о сообщать', np.float64(0.00020736988915804155)),
 ('по слово', np.float64(0.00017559151335825102)),
 ('по данные', np.float64(0.0001726703722593569)),
 ('ссылка на', np.float64(0.00016768222595450402)),
 ('президент россия', np.float64(0.00016495267564615944)),
 ('на сайт', np.float64(0.0001

In [21]:
sorted(t_scores_100.items(), key=lambda x: x[1], reverse=True)[:50]

[('по тема', np.float64(29.034779342452346)),
 ('материал по', np.float64(28.971661083100336)),
 ('один из', np.float64(20.641832991948036)),
 ('по слово', np.float64(19.904789227329328)),
 ('в год', np.float64(18.689342637493702)),
 ('о сообщать', np.float64(18.03641982032943)),
 ('в время', np.float64(15.082028065512983)),
 ('в пхенчхан', np.float64(13.183645299788584)),
 ('по данные', np.float64(12.830766146341503)),
 ('олимпийский игра', np.float64(12.62361323219482)),
 ('на сайт', np.float64(12.494492351663348)),
 ('игра в', np.float64(11.603844664385685)),
 ('в тот', np.float64(11.463002451742444)),
 ('российский спортсмен', np.float64(11.329912708166699)),
 ('в частность', np.float64(11.327931558862113)),
 ('в россия', np.float64(11.074194123249194)),
 ('владимир путин', np.float64(10.893433935592524)),
 ('тот число', np.float64(10.878861844865753)),
 ('в результат', np.float64(10.863932539111818)),
 ('ссылка на', np.float64(10.85465083430653)),
 ('участие в', np.float64(10.8496

Попробуем теперь избавиться от "странных" слов, получившихся из-за их склейки. PyMorphy2 помечает их, помещая данные о том, как проводился разбор в свойство `methods_stack`. Если слово было разобрано по словарю, будет использован `DictionaryAnalyzer`. Одно из многих значений для предсказанных слов - `FakeDictionary`. 

In [22]:
[type(x[0]) for x in morph.parse('русский')[0].methods_stack]

[pymorphy3.units.by_lookup.DictionaryAnalyzer]

In [23]:
[type(x[0]) for x in morph.parse('январярусский')[0].methods_stack]

[pymorphy3.units.by_analogy.KnownSuffixAnalyzer.FakeDictionary,
 pymorphy3.units.by_analogy.KnownSuffixAnalyzer]

In [25]:
pymorphy3.units.by_analogy.KnownSuffixAnalyzer.FakeDictionary in [type(x[0]) for x in morph.parse('январярусский')[0].methods_stack]

True

Перепишем функцию морфологического анализа так, чтобы она отсеивала несловарные слова (или хотя бы их часть).

In [26]:
imp_POS = ['ADJF', 'ADJS', 'NOUN', 'VERB', 'PRTF', 'PRTS', 'GRND', 'PREP']


def normalizePymorphy2(text):
    tokens = re.findall('[а-яёА-ЯЁ]+-[а-яёА-ЯЁ]+-[а-яёА-ЯЁ]+|[а-яёА-ЯЁ]+-[а-яёА-ЯЁ]+|[а-яёА-ЯЁ]+', text)
    words = []
    for t in tokens:
        pv = morph.parse(t)
        # Здесь мы проверяем, что слово было предсказано словарем.
        if str(pv[0].tag.POS) in imp_POS and \
          pymorphy3.units.by_lookup.DictionaryAnalyzer in [type(x[0]) for x in pv[0].methods_stack]:
            words.append(pv[0].normal_form)
            
    return words    


In [27]:
# lenta_news = [' '.join(normalizePymorphy2(news)) for news in tqdm(lenta_news.split(' '))]
# habr_news = [' '.join(normalizePymorphy2(news)) for news in tqdm(habr_news).split(' ')]

In [28]:
cntv12 = CountVectorizer(token_pattern="[а-яёА-ЯЁ]+-[а-яёА-ЯЁ]+-[а-яёА-ЯЁ]+|[а-яёА-ЯЁ]+-[а-яёА-ЯЁ]+|[а-яёА-ЯЁ]+", 
                        ngram_range=(1, 1))
cntv22 = CountVectorizer(token_pattern="[а-яёА-ЯЁ]+-[а-яёА-ЯЁ]+-[а-яёА-ЯЁ]+|[а-яёА-ЯЁ]+-[а-яёА-ЯЁ]+|[а-яёА-ЯЁ]+", 
                        ngram_range=(2, 2))

In [29]:
cnt_lenta_12 = cntv12.fit_transform(['\n'.join(lenta_news)])
cnt_lenta_22 = cntv22.fit_transform(['\n'.join(lenta_news)])

In [30]:
mis, t_scores = calc_MI_tscore(cnt_lenta_12, cnt_lenta_22, cntv12, cntv22, 10)

100%|█████████████████████████████████| 139120/139120 [00:51<00:00, 2724.51it/s]


In [31]:
sorted(mis.items(), key=lambda x: x[1], reverse=True)[:50]

[('январярусский смертьзакать', np.float64(0.1)),
 ('январясадизм какой-тобдсм-маска', np.float64(0.1)),
 ('февралязолоть стволыонить', np.float64(0.1)),
 ('упалокто ронять', np.float64(0.08333333333333333)),
 ('демьян кудрявцев', np.float64(0.08333333333333333)),
 ('возрождать сверхзвуковой', np.float64(0.07692307692307693)),
 ('рен тв', np.float64(0.07692307692307693)),
 ('сверхзвуковой ракетоносец', np.float64(0.06993006993006994)),
 ('рекс тиллерсон', np.float64(0.06666666666666667)),
 ('раюдин юсуфов', np.float64(0.06666666666666667)),
 ('шахабас шахов', np.float64(0.06666666666666667)),
 ('исаев раюдин', np.float64(0.0625)),
 ('какой-тобдсм-маска суровый', np.float64(0.058823529411764705)),
 ('февралякаждый своероссия', np.float64(0.058823529411764705)),
 ('оливковый ветвь', np.float64(0.05803571428571429)),
 ('суровый госпожа', np.float64(0.053475935828877004)),
 ('алина загитов', np.float64(0.04807692307692308)),
 ('шамиль исаев', np.float64(0.046875)),
 ('саудовский аравия', n

In [32]:
sorted(t_scores.items(), key=lambda x: x[1], reverse=True)[:50]

[('по тема', np.float64(29.034779342452346)),
 ('материал по', np.float64(28.971661083100336)),
 ('один из', np.float64(20.641832991948036)),
 ('по слово', np.float64(19.904789227329328)),
 ('в год', np.float64(18.689342637493702)),
 ('о сообщать', np.float64(18.03641982032943)),
 ('в время', np.float64(15.082028065512983)),
 ('в пхенчхан', np.float64(13.183645299788584)),
 ('по данные', np.float64(12.830766146341503)),
 ('олимпийский игра', np.float64(12.62361323219482)),
 ('на сайт', np.float64(12.494492351663348)),
 ('игра в', np.float64(11.603844664385685)),
 ('в тот', np.float64(11.463002451742444)),
 ('российский спортсмен', np.float64(11.329912708166699)),
 ('в частность', np.float64(11.327931558862113)),
 ('в россия', np.float64(11.074194123249194)),
 ('владимир путин', np.float64(10.893433935592524)),
 ('тот число', np.float64(10.878861844865753)),
 ('в результат', np.float64(10.863932539111818)),
 ('ссылка на', np.float64(10.85465083430653)),
 ('участие в', np.float64(10.8496

Расмотрим теперь еще одну метрику - странность (weirdness). Для ее вычисления требуется рассчитать частоты сочетаний выбранной длинны в тематической коллекции и в еще одной - контрастивной, специально подобранном из другой предметной области. После этого рассчитывается отношение частот в тематическом и контрастивной коллекции. Для того, чтобы не делить на 0, применим сглаживание Лагранжа, то есть прибавим 1 к знаменателю.

Так как термины предметной области будут встречаться чаще в тематической коллекции, то самые большие значения будут у слов, похожих на термины. Верно и обратное, так как термины контрастивной коллекции чаще встречаются в ней, то близкие к 0 значения будут означать термины контрастивной коллеции.

In [33]:
hcntv1 = CountVectorizer(token_pattern="[а-яёА-ЯЁ]+-[а-яёА-ЯЁ]+-[а-яёА-ЯЁ]+|[а-яёА-ЯЁ]+-[а-яёА-ЯЁ]+|[а-яёА-ЯЁ]+", 
                         ngram_range=(1, 1))
hcntv2 = CountVectorizer(token_pattern="[а-яёА-ЯЁ]+-[а-яёА-ЯЁ]+-[а-яёА-ЯЁ]+|[а-яёА-ЯЁ]+-[а-яёА-ЯЁ]+|[а-яёА-ЯЁ]+", 
                         ngram_range=(2, 2))

In [34]:
cnt_habr_1 = hcntv1.fit_transform(['\n'.join(habr_news)])
cnt_habr_2 = hcntv2.fit_transform(['\n'.join(habr_news)])

In [35]:
weir_1 = {}
weir_2 = {}
for word, index in tqdm(hcntv1.vocabulary_.items()):
    if word in cntv1.vocabulary_.keys():
        w = cnt_habr_1[0, index] / (cnt_lenta_1[0, cntv1.vocabulary_[word]] + 1)
    else:
        w = cnt_habr_1[0, index]
    weir_1[word] = w
    
for word, index in tqdm(hcntv2.vocabulary_.items()):
    if word in cntv2.vocabulary_.keys():
        w = cnt_habr_2[0, index] / (cnt_lenta_2[0, cntv2.vocabulary_[word]] + 1)
    else:
        w = cnt_habr_2[0, index]
    weir_2[word] = w

100%|███████████████████████████████████| 73353/73353 [00:17<00:00, 4151.18it/s]


In [37]:
sorted(weir_1.items(), key=lambda x: x[1], reverse=True)[:50]

[('шрифт', np.int64(162)),
 ('самохвалов', np.int64(153)),
 ('граф', np.int64(130)),
 ('настройка', np.int64(96)),
 ('евтеев', np.int64(90)),
 ('плагин', np.int64(82)),
 ('браузер', np.int64(72)),
 ('её', np.int64(68)),
 ('метрика', np.int64(64)),
 ('префикс', np.int64(57)),
 ('фреймворк', np.int64(52)),
 ('интерфейс', np.float64(50.5)),
 ('оптимизация', np.int64(50)),
 ('бурладянин', np.int64(50)),
 ('шаблон', np.int64(47)),
 ('скрипт', np.float64(46.5)),
 ('стек', np.int64(45)),
 ('хабра', np.int64(45)),
 ('текстура', np.int64(45)),
 ('трафаретный', np.int64(45)),
 ('линейный', np.int64(44)),
 ('буфер', np.int64(43)),
 ('строка', np.float64(42.75)),
 ('построение', np.int64(42)),
 ('переменный', np.float64(38.0)),
 ('тестировщик', np.int64(37)),
 ('актор', np.int64(37)),
 ('автоматизация', np.int64(35)),
 ('буква', np.float64(34.5)),
 ('вершина', np.float64(34.333333333333336)),
 ('нейронный', np.int64(33)),
 ('лямбда', np.int64(31)),
 ('классификация', np.int64(30)),
 ('преобразован

In [38]:
sorted(weir_2.items(), key=lambda x: x[1], reverse=True)[-50:]

[('произойти в', np.float64(0.03278688524590164)),
 ('по информация', np.float64(0.03225806451612903)),
 ('цена на', np.float64(0.03225806451612903)),
 ('быть сбить', np.float64(0.03225806451612903)),
 ('россия о', np.float64(0.03225806451612903)),
 ('в период', np.float64(0.03225806451612903)),
 ('счёт в', np.float64(0.03125)),
 ('в сообщение', np.float64(0.03125)),
 ('начало год', np.float64(0.03125)),
 ('за нарушение', np.float64(0.03125)),
 ('в январь', np.float64(0.031007751937984496)),
 ('год по', np.float64(0.030303030303030304)),
 ('в игра', np.float64(0.029411764705882353)),
 ('часть статья', np.float64(0.029411764705882353)),
 ('на борт', np.float64(0.02857142857142857)),
 ('новость в', np.float64(0.028037383177570093)),
 ('борьба с', np.float64(0.027777777777777776)),
 ('москва в', np.float64(0.02702702702702703)),
 ('опубликовать на', np.float64(0.02702702702702703)),
 ('пройти в', np.float64(0.02702702702702703)),
 ('в эфир', np.float64(0.02702702702702703)),
 ('год о', np

Если вам нужны термины длиннее, чем 2 слова,можно использовать [c-value](https://personalpages.manchester.ac.uk/staff/sophia.ananiadou/ijodl2000.pdf), которая определяет когда следует остановиться в цепочке слов.